In [1]:
!uv pip install openai tiktoken pydantic tenacity

Using Python 3.11.15 environment at: /Users/furqat/Desktop/ai-engineering/.venv
Checked 4 packages in 34ms


In [ ]:
from openai import OpenAI

In [18]:

client = OpenAI()
MODEL = "gpt-6-luna"

In [19]:
from openai.types import model
responses = client.responses.create(
    model=MODEL,
    input="salom"
)


In [ ]:
print(responses.model_dump_json(indent=2))

In [15]:
history = []

In [16]:
def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    res = client.responses.create(
        model=MODEL,
        instructions="You are a gamer",
        input=history
    )
    history.append({"role": "assistant", "content": res.output_text})

    return res.output_text

In [ ]:
print(chat("Salom ismim Furqat"))
print(chat("ismim nima?"))

In [22]:
#temperature
from openai import BadRequestError

try:
    r = client.responses.create(
        model=MODEL,
        temperature=0.2,
        input="Salom"
    )
    print("ishladi: ", r.output_text)
except BadRequestError as e:
    print("ishlamadi: ", e.message)

ishlamadi:  Error code: 400 - Unsupported parameter: 'temperature' is not supported with this model.


In [27]:
#max tokens
r = client.responses.create(
    model=MODEL,
    input="Uzbeksitan tarixi xaqida yoz",
    max_output_tokens=3000
)

print(r.status)
print(r.incomplete_details)
print(repr(r.output_text))

completed
None
'## O‘zbekiston tarixi\n\nO‘zbekiston hududi qadimdan Markaziy Osiyoning muhim sivilizatsiya va savdo markazlaridan biri bo‘lib kelgan. Bu yerda dehqonchilik, hunarmandchilik va shahar madaniyati rivojlangan, Buyuk ipak yo‘li esa Sharq va G‘arbni bog‘lagan.\n\n### Qadimgi davr\n\nHozirgi O‘zbekiston hududida qadimda **So‘g‘diyona, Baqtriya va Xorazm** kabi tarixiy o‘lkalar mavjud bo‘lgan. Miloddan avvalgi VI–IV asrlarda bu yerlarning ayrim qismlari Ahamoniylar imperiyasi tarkibiga kirgan. Miloddan avvalgi IV asrda Aleksandr Makedonskiy Markaziy Osiyoga yurish qilgan. Keyingi asrlarda hududda Yunon-Baqtriya va Kushon podsholiklari kabi davlatlar hukmronlik qilgan.\n\n### O‘rta asrlar\n\nVIII asrda arablar Movarounnahrni egallab, islom dini keng tarqala boshlagan. IX–X asrlarda Buxoroni markaz qilgan **Somoniylar davlati** yuksaldi. Bu davrda ilm-fan, adabiyot va me’morchilik rivojlandi. Keyinchalik hudud Qoraxoniylar va Xorazmshohlar kabi davlatlar tarkibida bo‘ldi.\n\nXI

In [29]:
import tiktoken

enc = tiktoken.get_encoding('o200k_base')

for text in ["How are you?", "Qalaysan, ishlaring yaxshimi?", "Как дела?"]:
    tokens = enc.encode(text)
    print(len(tokens), text)
    print([enc.decode([t]) for t in tokens])

4 How are you?
['How', ' are', ' you', '?']
10 Qalaysan, ishlaring yaxshimi?
['Q', 'al', 'aysan', ',', ' ish', 'laring', ' yax', 'sh', 'imi', '?']
3 Как дела?
['Как', ' дела', '?']


In [30]:
PRICE_IN = 0.10
PRICE_OUT = 0.50

def cost_usd(usage) -> float:
    return (usage.input_tokens * PRICE_IN + usage.output_tokens * PRICE_OUT) / 1_000_000

r = client.responses.create(
    model=MODEL,
    input="Rekursiya nima?"
)

print(r.usage.input_tokens, r.usage.output_tokens, f"{cost_usd(r.usage):.6f}")

11 173 0.000088


In [31]:
#error handling
from openai import NotFoundError, AuthenticationError

try:
    OpenAI(api_key="sk-worng").responses.create(model=MODEL, input="Salom")
except AuthenticationError as e:
    print("Xatolik: ", e.message)

try:
    client.responses.create(model="wrong-model", input="salom")
except NotFoundError as e:
    print("Xatolik: ", e.message)

Xatolik:  Error code: 401 - Incorrect API key provided: sk-worng. You can find your API key at https://platform.openai.com/account/api-keys.
Xatolik:  Error code: 404 - The model `wrong-model` does not exist or you do not have access to it.


In [33]:
#streaming
stream = client.responses.create(
    model=MODEL,
    input="Top features of python",
    stream=True
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print("\n\nTugadi")

### Top features of Python

- **Readable syntax:** Its clear, concise style makes code easier to write and understand.
- **Easy to learn:** Python is often recommended for beginners.
- **Interpreted:** You can run code without a separate compilation step.
- **Dynamically typed:** Variables don’t need explicit type declarations.
- **Multi-paradigm:** Supports procedural, object-oriented, and functional programming.
- **Large standard library:** Includes tools for tasks such as file handling, networking, and data processing.
- **Extensive ecosystem:** Many third-party packages support areas like web development, automation, data science, and AI.
- **Cross-platform:** Python programs can run on Windows, macOS, and Linux.
- **Automatic memory management:** Python handles much of the memory allocation and cleanup for you.
- **Extensible:** It can work with code and libraries written in languages such as C and C++.

Tugadi


In [46]:
from openai.types.decision_create_params import QuestionQuestionParamPredicate
#structured output
from pydantic import BaseModel, Field
from typing import Literal

class QuizQuestion(BaseModel):
    question: str
    options: list[str] = Field(description="4 ta variant")
    correct_index: int = Field(ge=0, le=3)
    explanation: str

class Quiz(BaseModel):
    topic: str
    difficulty: Literal["easy", "medium", "hard"]
    questions: list[QuizQuestion]

res = client.responses.parse(
        model=MODEL,
        instructions="Berilgan matn asosida o'zbek tilida test tuz. 5 ta",
        input="Python'da list o'zgaruvchan (mutable), tuple esa o'zgarmas (immutable). ...",
        text_format=Quiz
    )

quiz = res.output_parsed

print(quiz.topic, quiz.difficulty)

print(quiz.model_dump)

Python'da list va tuple easy
<bound method BaseModel.model_dump of Quiz(topic="Python'da list va tuple", difficulty='easy', questions=[QuizQuestion(question="Python'da qaysi ma'lumot turi o'zgaruvchan (mutable) hisoblanadi?", options=['tuple', 'list', 'str', 'int'], correct_index=1, explanation="List o'zgaruvchan bo'lib, uning elementlarini yaratgandan keyin o'zgartirish mumkin."), QuizQuestion(question="Python'da tuple qanday xususiyatga ega?", options=["O'zgaruvchan", 'Faqat son saqlaydi', "O'zgarmas", 'Kalit-qiymat juftliklarini saqlaydi'], correct_index=2, explanation="Tuple o'zgarmas (immutable): yaratilgandan keyin uning elementlarini o'zgartirib bo'lmaydi."), QuizQuestion(question="Quyidagi qaysi amal list uchun mumkin, lekin tuple elementlariga bevosita qo'llanmaydi?", options=["Elementni indeks orqali o'qish", "Elementni indeks orqali o'zgartirish", 'Elementlar sonini aniqlash', "Elementlar bo'ylab yurish"], correct_index=1, explanation="List elementini indeks orqali o'zgartir

In [49]:
#flashcard generation
class Flashcard(BaseModel):
    front: str
    back: str


class Collection(BaseModel):
    name: str
    cards: list[Flashcard]


flashcard = client.responses.parse(
    model=MODEL,
    input="Generate 10 flashcards spanish-uzbek langueage flashcards",
    text_format=Collection
)

collection = flashcard.output_parsed
print(collection.name)

for card in collection.cards:
    print(card.front, "->", card.back)

Spanish–Uzbek Language Flashcards
Hola -> Salom
Gracias -> Rahmat
Por favor -> Iltimos
Sí -> Ha
No -> Yo‘q
Agua -> Suv
Comida -> Ovqat
Amigo / Amiga -> Do‘st
¿Cómo estás? -> Qalaysiz?
Adiós -> Xayr


In [52]:
import asyncio, time
from openai import AsyncOpenAI

aclient = AsyncOpenAI(timeout=30.0, max_retries=3)

async def summarize(topic: str) -> str:
    r = await aclient.responses.create(
        model=MODEL,
        input=f"{topic} haqida bir jumla yoz"
    )

    return r.output_text

topics = ["list", "tuple", "set", "generator", "dict"]

#ketma ket
start = time.perf_counter()
for topic in topics:
    result = await summarize(topic)
    print(result)

print(f"Ketma-ket: {time.perf_counter() - start:.1f}s")

#parallel
start = time.perf_counter()
results = await asyncio.gather(*(summarize(topic) for topic in topics))

print(f"Parallel: {time.perf_counter() - start:.1f}s")

List — bir nechta elementlarni tartib bilan saqlashga mo‘ljallangan ma’lumotlar tuzilmasi.
Tuple — bir nechta qiymatni tartiblangan va o‘zgarmas to‘plam sifatida saqlaydigan ma’lumot turi.
To‘plam — muayyan belgiga ega bo‘lgan elementlar majmuasidir.
Generator mexanik energiyani elektr energiyasiga aylantiradigan qurilmadir.
Python’da `dict` — kalit va qiymatlarni juftlab saqlaydigan ma’lumotlar tuzilmasi.
Ketma-ket: 10.6s
Parallel: 3.2s


In [57]:
doc = open("laravel_book.txt", encoding="utf-8").read()

def ask_doc(question: str):
    r = client.responses.create(
        model=MODEL,
        instructions="Faqat berilgan hujjat asosida javob ber.",
        input=[
            {"role": "user", "content": f"Hujjat:\n{doc}"},
            {"role": "user", "content": f"Savol:\n{question}"}
        ],
        prompt_cache_key="laravel_book"
    )

    u = r.usage
    print(f"input={u.input_tokens}, keshdan={u.input_tokens_details.cached_tokens}")

    return r.output_text


print(ask_doc("laravel nima?"))
print(ask_doc("Laravelda routing nima?"))

input=3233, keshdan=3230
Laravel — PHP asosida yaratilgan veb-ilovalar freymvorki. U ilova yaratish uchun tuzilma va boshlang‘ich asosni beradi, shunda dasturchi kerakli funksiyalarni yaratishga e’tibor qaratishi mumkin. Unda, jumladan, ma’lumotlar bazasi bilan ishlash, navbatlar, rejalashtirilgan vazifalar va testlash imkoniyatlari mavjud.
input=3234, keshdan=3231
Hujjatda routing Laravelning asosiy vazifalaridan biri sifatida tilga olinadi: Laravel so‘rovlarni ilovangizga yo‘naltirishi mumkin. Routing haqida batafsil ta’rif hujjatda berilmagan.
